In [2]:
# Import libraries
import pandas as pd 
import numpy as np 
import math 
import statistics 
import matplotlib.pyplot as plt 

In [3]:
### Initial parse and inspection ###

# Import and inspect ES_2020-2025.csv 
raw_df = pd.read_csv(r".\Signal_data\ES_2020-2025.csv", sep=";", engine="python") 

# Remove first row of NaN data
raw_df = raw_df.iloc[1:].reset_index(drop=True) 

# Remove the imported but empty last column 
raw_df = raw_df.iloc[:,:-1]

# Display current header 
raw_df.head() 

,Quarter,Model_Classification_ID,Entry_Date_ISO,ES_Entry_Price,Direction,Exit_Date_ISO,ES_Exit_Price
0,q3-20,q3-20_1,2020-08-10,"3347,00",-1.0,2020-08-11,"3350,25"
1,q3-20,q3-20_2,2020-08-11,"3350,25",1.0,2020-08-19,"3388,75"
2,q3-20,q3-20_3,2020-08-19,"3388,75",-1.0,2020-08-20,"3370,25"
3,q3-20,q3-20_4,2020-08-20,"3370,25",1.0,2020-08-28,"3488,00"
4,q4-20,q4-20_1,2020-09-25,"3243,50",1.0,2020-09-28,"3291,00"


In [4]:
### Edit column value types and tidy formatting ###
 
# Change column names
columns = ["Quarter", "Model_Classification_ID", "Entry_Date", "Entry_Price", "Signal", "Exit_Date", "Exit_Price"]
raw_df.columns = columns 

# Change prices to floats 
raw_df["Entry_Price"] = raw_df["Entry_Price"].str.replace(",", ".").astype(float) 
raw_df["Exit_Price"] = raw_df["Exit_Price"].str.replace(",", ".").astype(float)

# Change date strings to datetime 
raw_df["Entry_Date"] = pd.to_datetime(raw_df["Entry_Date"])
raw_df["Exit_Date"] = pd.to_datetime(raw_df["Exit_Date"])

# Review data formatting
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Quarter                  117 non-null    object        
 1   Model_Classification_ID  117 non-null    object        
 2   Entry_Date               117 non-null    datetime64[ns]
 3   Entry_Price              117 non-null    float64       
 4   Signal                   117 non-null    float64       
 5   Exit_Date                117 non-null    datetime64[ns]
 6   Exit_Price               117 non-null    float64       
dtypes: datetime64[ns](2), float64(3), object(2)
memory usage: 6.5+ KB


In [5]:
raw_df.head() 

,Quarter,Model_Classification_ID,Entry_Date,Entry_Price,Signal,Exit_Date,Exit_Price
0,q3-20,q3-20_1,2020-08-10,3347.00,-1.0,2020-08-11,3350.25
1,q3-20,q3-20_2,2020-08-11,3350.25,1.0,2020-08-19,3388.75
2,q3-20,q3-20_3,2020-08-19,3388.75,-1.0,2020-08-20,3370.25
3,q3-20,q3-20_4,2020-08-20,3370.25,1.0,2020-08-28,3488.00
4,q4-20,q4-20_1,2020-09-25,3243.50,1.0,2020-09-28,3291.00


**Next Steps**:
- Analyse number of trades, direction of trade and hit rate.
- Analyse typical returns and volatility
- Acquire historical data for each instrument during trade windows to do daily settlement/returns analysis

In [ ]:
# Number of trades
Num_trades = len(raw_df) 
print(f"No. of trades: {Num_trades}")

# Add trade return and hit columns
raw_df["Returns"] = (raw_df["Exit_Price"] - raw_df["Entry_Price"])*raw_df["Signal"]
raw_df["Hit"] = (raw_df["Returns"] > 0).astype(int)

# Count number of successful and unsuccessful trade
Num_hits = sum(raw_df.Hit) 
print(f"No. of hits: {Num_hits}")
print(f"Hit rate: {(Num_hits/Num_trades)*100:.2f}%")


No. of trades: 117
No. of hits: 67
Hit rate: 57.26%


,Quarter,Model_Classification_ID,Entry_Date,Entry_Price,Signal,Exit_Date,Exit_Price,Returns,Hit
0,q3-20,q3-20_1,2020-08-10,3347.00,-1.0,2020-08-11,3350.25,-3.25,0
1,q3-20,q3-20_2,2020-08-11,3350.25,1.0,2020-08-19,3388.75,38.50,1
2,q3-20,q3-20_3,2020-08-19,3388.75,-1.0,2020-08-20,3370.25,18.50,1
3,q3-20,q3-20_4,2020-08-20,3370.25,1.0,2020-08-28,3488.00,117.75,1
4,q4-20,q4-20_1,2020-09-25,3243.50,1.0,2020-09-28,3291.00,47.50,1


In [21]:
# Show current dataframe
raw_df.tail()

,Quarter,Model_Classification_ID,Entry_Date,Entry_Price,Signal,Exit_Date,Exit_Price,Returns,Hit
112,s4-25,s4-25_5,2025-11-21,6560.00,-1.0,2025-11-24,6654.75,-94.75,0
113,s4-25,s4-25_6,2025-11-24,6654.75,1.0,2025-12-02,6829.00,174.25,1
114,s4-25,s4-25_7,2025-12-02,6829.00,-1.0,2025-12-04,6866.00,-37.00,0
115,s4-25,s4-25_8,2025-12-04,6866.00,1.0,2025-12-11,6887.00,21.00,1
116,s4-25,s4-25_9,2025-12-11,6887.00,-1.0,2025-12-18,6739.25,147.75,1


In [11]:
import yfinance as yf 

es = yf.download("ES=F", start="2020-08-10", end="2020-08-20", auto_adjust="False") 

print(es)

[*********************100%***********************]  1 of 1 completed

Price         Close     High      Low     Open   Volume
Ticker         ES=F     ES=F     ES=F     ES=F     ES=F
Date                                                   
2020-08-10  3352.75  3357.25  3329.00  3347.00  1201751
2020-08-11  3330.00  3379.00  3319.50  3350.25  1733040
2020-08-12  3370.00  3382.50  3326.25  3339.50  1408718
2020-08-13  3367.75  3382.00  3357.50  3367.75  1205758
2020-08-14  3361.50  3380.50  3350.00  3369.25  1084771
2020-08-17  3379.75  3382.75  3364.75  3366.00   759304
2020-08-18  3387.00  3390.75  3365.25  3379.25  1019552
2020-08-19  3372.75  3395.75  3365.50  3388.75  1368420
